In [0]:
# Upgrade Databricks SDK to the latest version and restart Python to see updated packages
%pip install --upgrade databricks-sdk==0.70.0
%restart_python

from databricks.sdk.service.jobs import JobSettings as Job


IAP_job = Job.from_dict(
    {
        "name": "IAP_job",
        "email_notifications": {
            "on_failure": [
                "rahulpatel@cyntexa.com",
            ],
            "no_alert_for_skipped_runs": True,
        },
        "notification_settings": {
            "no_alert_for_skipped_runs": True,
            "no_alert_for_canceled_runs": True,
        },
        "schedule": {
            "quartz_cron_expression": "48 0 6 * * ?",
            "timezone_id": "Asia/Calcutta",
            "pause_status": "PAUSED",
        },
        "tasks": [
            {
                "task_key": "bronze_ingetion",
                "notebook_task": {
                    "notebook_path": "/Workspace/Users/patelrahul2614@gmail.com/IAP-MainExam/TPCH-Sales&SupplyChain/01_bronze",
                    "base_parameters": {
                        "job_run_id": "{{run_id}}",
                    },
                    "source": "WORKSPACE",
                },
            },
            {
                "task_key": "silver_transformation",
                "depends_on": [
                    {
                        "task_key": "bronze_ingetion",
                    },
                ],
                "notebook_task": {
                    "notebook_path": "/Workspace/Users/patelrahul2614@gmail.com/IAP-MainExam/TPCH-Sales&SupplyChain/02_silver",
                    "base_parameters": {
                        "job_run_id": "{{run_id}}",
                    },
                    "source": "WORKSPACE",
                },
            },
            {
                "task_key": "gold_model",
                "depends_on": [
                    {
                        "task_key": "silver_transformation",
                    },
                ],
                "notebook_task": {
                    "notebook_path": "/Workspace/Users/patelrahul2614@gmail.com/IAP-MainExam/TPCH-Sales&SupplyChain/03_gold",
                    "base_parameters": {
                        "job_run_id": "{{run_id}}",
                    },
                    "source": "WORKSPACE",
                },
            },
            {
                "task_key": "data_quality_check",
                "depends_on": [
                    {
                        "task_key": "gold_model",
                    },
                ],
                "notebook_task": {
                    "notebook_path": "/Workspace/Users/patelrahul2614@gmail.com/IAP-MainExam/TPCH-Sales&SupplyChain/04_dataquality",
                    "base_parameters": {
                        "job_run_id": "{{run_id}}",
                    },
                    "source": "WORKSPACE",
                },
            },
        ],
        "queue": {
            "enabled": True,
        },
        "parameters": [
            {
                "name": "catalog",
                "default": "tpch",
            },
        ],
        "performance_target": "PERFORMANCE_OPTIMIZED",
    }
)

from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
w.jobs.reset(new_settings=IAP_job, job_id=177782946141925)
# or create a new job using: w.jobs.create(**IAP_job.as_shallow_dict())


![image_1792659638879.png](./image_1792659638879.png "image_1792659638879.png")
![image_1792661396221.png](./image_1792661396221.png "image_1792661396221.png")

## Scheduling, Deploying & Monitoring the Pipeline in Production

### 1. Schedule the Job
The **`IAP_job`** orchestrates the whole pipeline end-to-end with four sequential tasks — **Bronze** (raw ingestion) → **Silver** (cleansing & transformation) → **Gold** (curated analytical models) → **Data Quality Check** (validates gold tables). A **quartz cron** schedule (`48 0 6 * * ?` in `Asia/Calcutta`) fires it every day at **6:00 AM**. It starts **paused** — just unpause it from the Jobs UI or via the SDK once you're ready to go live.

### 2. Deploy to Production with Databricks Asset Bundles (DABS)
Instead of clicking through the UI every time, you can ship this pipeline to production using **Databricks Asset Bundles**. All the job configuration — tasks, schedule, notifications, parameters — lives as a **Python** or **YAML** file inside a resource folder. You can build the bundle straight from the Databricks UI: the job definition can be exported as Python or YAML code, dropped into the bundle's `resources/` directory, and then deployed with a single `databricks bundle deploy` command. This makes it version-controlled, repeatable, and easy to promote across dev, staging, and prod environments.

### 3. Monitor the Pipeline
- **Jobs UI** — check run status, the task DAG, duration, and logs for each task.
- **Notifications** — set up alerts on failure or success to your preferred destination: **email**, **Slack**, **Microsoft Teams**, or any webhook-based system — all configurable from the job's notification settings.
- **Run history** — track successes, failures, and retries over time.
- **Data Quality gate** — if quality rules fail, the job fails and notifications fire automatically.
- **Job parameters** — the `catalog` parameter (`tpch`) is passed to every notebook, so switching environments is just a parameter change.

### 4. Production Readiness Checklist
| Step | Action |
|-----|--------|
| Deploy the bundle | `databricks bundle deploy` from the resource folder |
| Unpause schedule | Jobs UI → Edit → Unpause |
| Verify notifications | Confirm email / Slack / Teams destinations are correct |
| Test a manual run | Trigger the job once to validate end-to-end |
| Monitor first scheduled run | Check run history & task logs |
| Review data quality results | Inspect the quality check notebook output |